# N-CMAPSS DS05 — BO and PSO on the matched protocol

## Where v7 left us

The factorial closed the gap to MSTSDN almost entirely with protocol alone:

| configuration | capped-55 RMSE | skill |
|---|---|---|
| our old setup (18 features, snapshot, uncapped) | 15.157 +/- 5.93 | +0.177 |
| matched protocol (28 sensors, T=30, cap 55) | **5.137 +/- 0.43** | **+0.721** |
| MSTSDN published | 3.95 | +0.786 |

and the important structural finding was the **interaction**: the virtual sensors and the time window
carry the same information. Either alone recovers most of it (-7.85 and -9.83); the second one added
on top buys +0.23 and -1.75. Our snapshot representation was starving the model of temporal context.

**1.187 RMSE remains.** This notebook asks how much of it is reachable by hyperparameter search on
our architecture, before we consider whether the rest requires theirs.

## Why this search should behave better than the earlier ones

Every previous search on this dataset ran on top of the snapshot representation — the cell v7 showed
is *unstable*, sd 5.93 across seeds. A search cannot rank configurations when the noise on each
evaluation is larger than the differences between them; that is the flat-optimum result we kept
hitting. On the matched protocol the seed sd is 0.43. The signal-to-noise is roughly fourteen times
better, so for the first time the objective may actually be able to discriminate.

That is a hypothesis, not an assumption, so the notebook measures it directly: **Spearman(validation,
test) across all trials**. If it is near zero the search cannot work here regardless of budget, and we
will have said so with evidence.

(For the record: the +0.905 Spearman I reported earlier for cross-cycle was a bimodal artifact of the
cycle-index feature being on or off. The honest value across 103 configs was **+0.276**.)

## Design rules carried over from FD003

- **One budget everywhere.** Search and final report train identically. No 30-epoch search feeding a
  60-epoch headline.
- **Per-trial test score recorded, never used for selection.** It exists only so we can see where
  validation and test diverge.
- **Baseline is in the population but is not the start point.** Both searches begin far away in the
  space, so if they converge back to the baseline region that is a finding, not a construction.
- **Two-stage selection.** Stage 1 ranks on one seed. Stage 2 re-decides among the shortlist on fresh
  seeds, so the winner is not just the luckiest draw. Argmin over noisy estimates is biased.
- **Matched head-to-head guard.** The winner only replaces the baseline if it beats it at identical
  seeds and identical budget.

## Reporting convention

Both target conventions are carried through every table: **raw** (our thesis numbering) and
**capped-55** (MSTSDN's). Selection uses capped validation, since the models train on capped targets.
Nothing already written becomes wrong; the benchmark comparison stays fair.

In [ ]:
import os, json, gc, time, random, math, itertools, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from scipy.stats import norm, spearmanr
from scipy.stats.qmc import Sobol
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

## Data — the winning cell of v7, unchanged

In [ ]:
SEED=42; SUB=50; VAL_UNITS=[5,6]; CAP=55; T=30
EPOCHS=100; PATIENCE=15
N_BO=60; N_PSO=60; SHORTLIST=5; N_SEEDS2=5
set_seeds(SEED)
def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Sd=read_sub(h['X_s_dev'],SUB); Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);   Yd=read_sub(h['Y_dev'],SUB)
    St=read_sub(h['X_s_test'],SUB);Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB);  Yt=read_sub(h['Y_test'],SUB)
Fd=np.hstack([Sd,Vd]); Ft=np.hstack([St,Vt])
ud,cd,yd=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
ut,ct,yt=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()

def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]
    bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

tr=~np.isin(ud,VAL_UNITS); va=np.isin(ud,VAL_UNITS)
sc=MinMaxScaler().fit(Fd[tr])
Fd2=sc.transform(Fd).astype(np.float32); Ft2=sc.transform(Ft).astype(np.float32)
Xtr,ytr_raw=windows(Fd2,ud,cd,yd,tr,T)
Xva,yva_raw=windows(Fd2,ud,cd,yd,va,T)
Xte,yte_raw=windows(Ft2,ut,ct,yt,np.ones(len(yt),bool),T)
ytr=np.minimum(ytr_raw,CAP); yva=np.minimum(yva_raw,CAP); yte_cap=np.minimum(yte_raw,CAP)
del Fd,Ft,Fd2,Ft2; gc.collect()
SHAPE=(T,Xtr.shape[2])
triv_raw=float(np.sqrt(mean_squared_error(yte_raw,np.full_like(yte_raw,ytr_raw.mean()))))
triv_cap=float(np.sqrt(mean_squared_error(yte_cap,np.full_like(yte_cap,ytr.mean()))))
print('shape %s | train %s | val %s | test %s'%(SHAPE,format(len(ytr),','),
      format(len(yva),','),format(len(yte_raw),',')))
print('trivial floors -> raw %.3f | capped-%d %.3f'%(triv_raw,CAP,triv_cap))
print('leakage check: val units %s, train units %s, overlap %s'
      %(VAL_UNITS,sorted(set(ud[tr])),sorted(set(ud[tr])&set(VAL_UNITS))))

## Search space — 13 hyperparameters

Every structural choice we have any freedom over, not just the safe few. The space is defined once,
as a unit cube, so BO and PSO explore exactly the same thing and their results are comparable.

In [ ]:
SPACE=[('n_conv','int',2,4), ('f1','int',24,192), ('f_growth','float',0.5,2.0),
       ('kernel','cat',[2,3,5,7]), ('use_pool','cat',[0,1]),
       ('n_lstm','int',1,2), ('lstm_units','int',24,192),
       ('dropout','float',0.0,0.40), ('l2reg','log',1e-7,1e-1),
       ('dense1','int',32,256), ('dense2','int',0,160),
       ('lr','log',1e-4,5e-3), ('batch','cat',[128,256,512,1024])]
D=len(SPACE); HPN=[s[0] for s in SPACE]

def decode(z):
    z=np.clip(np.asarray(z,float),0,1); out={}
    for zi,s in zip(z,SPACE):
        n,k=s[0],s[1]
        if k=='int':   out[n]=int(round(s[2]+zi*(s[3]-s[2])))
        elif k=='float':out[n]=float(s[2]+zi*(s[3]-s[2]))
        elif k=='log': out[n]=float(np.exp(np.log(s[2])+zi*(np.log(s[3])-np.log(s[2]))))
        else:
            opts=s[2]; out[n]=opts[min(int(zi*len(opts)),len(opts)-1)]
    return out
def encode(kw):
    z=[]
    for s in SPACE:
        n,k=s[0],s[1]; v=kw[n]
        if k=='int':   z.append((v-s[2])/(s[3]-s[2]))
        elif k=='float':z.append((v-s[2])/(s[3]-s[2]))
        elif k=='log': z.append((np.log(v)-np.log(s[2]))/(np.log(s[3])-np.log(s[2])))
        else:
            opts=s[2]; z.append((opts.index(v)+0.5)/len(opts))
    return np.clip(np.array(z,float),0,1)

BASE=dict(n_conv=4,f1=96,f_growth=1.15,kernel=3,use_pool=1,n_lstm=1,lstm_units=80,
          dropout=0.10,l2reg=1e-7,dense1=192,dense2=96,lr=1e-3,batch=512)
Z_BASE=encode(BASE)
print('search dimension:',D)
print('baseline encodes to:',np.round(Z_BASE,3))
assert all(abs(decode(Z_BASE)[k]-BASE[k])<1e-6 or decode(Z_BASE)[k]==BASE[k]
           for k in ('n_conv','kernel','lstm_units','batch','dense1')), 'encode/decode mismatch'
print('encode/decode round-trip OK')

In [ ]:
def build(kw,shape=SHAPE,seed=SEED):
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    ini=tf.keras.initializers.GlorotUniform(seed=seed)
    kk=dict(kernel_initializer=ini,kernel_regularizer=reg)
    inp=Input(shape=shape); c=inp
    for i in range(kw['n_conv']):
        f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
        c=Conv1D(f,kw['kernel'],padding='same',activation='relu',**kk)(c)
        if kw['dropout']>0: c=Dropout(kw['dropout'],seed=seed)(c)
        if kw['use_pool'] and i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=GlobalAveragePooling1D()(c)
    r=inp
    for j in range(kw['n_lstm']):
        r=LSTM(kw['lstm_units'],return_sequences=(j<kw['n_lstm']-1),**kk)(r)
    h=Concatenate()([cv,r])
    h=Dense(kw['dense1'],activation='relu',**kk)(h)
    if kw['dense2']>=16: h=Dense(kw['dense2'],activation='relu',**kk)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss='mse'); return m

TRIALS=[]
def evaluate(kw,seed=SEED,tag='',record=True):
    """One budget everywhere. Test scores recorded but NEVER returned to the optimiser."""
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    try:
        with tf.device(DEVICE):
            m=build(kw,seed=seed)
            m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=EPOCHS,batch_size=kw['batch'],verbose=0,
                  callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                             ReduceLROnPlateau('val_loss',factor=0.5,patience=PATIENCE//2,min_lr=1e-6)])
            pv=m.predict(Xva,batch_size=8192,verbose=0).ravel()
            pt=m.predict(Xte,batch_size=8192,verbose=0).ravel()
        val=float(np.sqrt(mean_squared_error(yva,pv)))
        t_cap=float(np.sqrt(mean_squared_error(yte_cap,np.clip(pt,0,CAP))))
        t_raw=float(np.sqrt(mean_squared_error(yte_raw,pt)))
        nprm=int(m.count_params())
    except Exception as e:
        print('   trial failed (%s) -> penalised'%type(e).__name__)
        val,t_cap,t_raw,nprm=1e3,np.nan,np.nan,0
    if not np.isfinite(val): val=1e3
    if record:
        TRIALS.append(dict(tag=tag,seed=seed,val=val,test_capped=t_cap,test_raw=t_raw,
                           params=nprm,secs=round(time.time()-t0,1),**kw))
    return val   # <-- selection signal is validation only

print('warm-up: baseline at one seed')
v0=evaluate(BASE,SEED,'warmup'); w=TRIALS[-1]
print('  val %.3f | test capped %.3f | test raw %.3f | %d params | %.0fs'
      %(v0,w['test_capped'],w['test_raw'],w['params'],w['secs']))
print('\nestimated search time: %.0f min for %d evaluations'%((N_BO+N_PSO)*w['secs']/60,N_BO+N_PSO))

## Bayesian optimisation

Gaussian process with a Matern 5/2 kernel and a white-noise term, Expected Improvement acquisition.

The initial design is Sobol, **rejected if it lands within 0.35 of the baseline in normalised
distance** — so the search genuinely starts far away. The baseline is appended to the observed set
after that design, which puts it in the population without letting it seed the search.

In [ ]:
def nrmsd(a,b):
    """per-dimension RMS distance in the unit cube; 0 = identical, 1 = opposite corners"""
    return float(np.linalg.norm(np.asarray(a)-np.asarray(b))/np.sqrt(D))

def far_design(n,z_ref,min_d=0.35,seed=SEED):
    eng=Sobol(d=D,scramble=True,seed=seed); pts=[]; tries=0
    while len(pts)<n and tries<50:
        tries+=1
        for c in eng.random(max(8,n*4)):
            if nrmsd(c,z_ref)>=min_d:
                pts.append(c)
                if len(pts)==n: break
        if len(pts)==n: break
    if len(pts)<n:            # never silently return a short design
        rs=np.random.RandomState(seed)
        while len(pts)<n:
            c=rs.rand(D)
            if nrmsd(c,z_ref)>=min_d: pts.append(c)
    return np.array(pts[:n])

N_INIT=14
Zi=far_design(N_INIT,Z_BASE)
assert len(Zi)==N_INIT
print('initial design: %d points | distance to baseline: min %.3f, mean %.3f (threshold 0.35)'
      %(len(Zi),min(nrmsd(z,Z_BASE) for z in Zi),np.mean([nrmsd(z,Z_BASE) for z in Zi])))

Zobs=[]; Yobs=[]
for i,z in enumerate(Zi):
    y=evaluate(decode(z),SEED,'bo_init'); Zobs.append(z); Yobs.append(y)
    print('  init %2d/%d  val %7.3f'%(i+1,N_INIT,y))
Zobs.append(Z_BASE.copy()); Yobs.append(v0)          # baseline joins the population here
print('  baseline injected into population: val %.3f'%v0)

def ei(mu,sd,best,xi=0.01):
    sd=np.maximum(sd,1e-9); imp=best-mu-xi; z=imp/sd
    return imp*norm.cdf(z)+sd*norm.pdf(z)

t0=time.time()
for it in range(N_BO-N_INIT):
    Za,Ya=np.array(Zobs),np.array(Yobs)
    ok=Ya<999; Zf,Yf=(Za[ok],Ya[ok]) if ok.sum()>=5 else (Za,Ya)
    gp=GaussianProcessRegressor(ConstantKernel(1.0)*Matern(length_scale=np.ones(D),nu=2.5)
                                +WhiteKernel(1e-2),normalize_y=True,n_restarts_optimizer=3,
                                random_state=SEED)
    gp.fit(Zf,Yf)
    cand=np.random.RandomState(SEED+it).rand(4000,D)
    mu,sd=gp.predict(cand,return_std=True)
    zbest=cand[int(np.argmax(ei(mu,sd,Yf.min())))]
    y=evaluate(decode(zbest),SEED,'bo'); Zobs.append(zbest); Yobs.append(y)
    print('  bo %2d/%d  val %7.3f  (best %7.3f)'%(it+1,N_BO-N_INIT,y,min(Yobs)))
print('BO done in %.1f min | best val %.3f'%((time.time()-t0)/60,min(Yobs)))
BO_Z,BO_Y=np.array(Zobs),np.array(Yobs)

## Particle swarm

Same space, same budget, same evaluator. **Particle 0 is the baseline** — so the baseline is inside
the swarm from the first iteration and the swarm cannot do worse than it by construction. Every other
particle starts far away, and velocities are initialised outward.

In [ ]:
SWARM=12; ITERS=N_PSO//SWARM
rng=np.random.RandomState(SEED+7)
P=far_design(SWARM,Z_BASE,seed=SEED+7)
P[0]=Z_BASE.copy()                                   # particle 0 is the baseline
V=(rng.rand(SWARM,D)-0.5)*0.30
pbest=P.copy(); pbest_y=np.full(SWARM,np.inf)
gbest=None; gbest_y=np.inf
w_hi,w_lo,c1,c2=0.85,0.35,1.5,1.5
t0=time.time()
for it in range(ITERS):
    w_in=w_hi-(w_hi-w_lo)*it/max(1,ITERS-1)
    for k in range(SWARM):
        y = v0 if (it==0 and k==0) else evaluate(decode(P[k]),SEED,'pso')
        if it==0 and k==0: TRIALS.append(dict(tag='pso',seed=SEED,val=v0,
            test_capped=w['test_capped'],test_raw=w['test_raw'],params=w['params'],
            secs=w['secs'],**BASE))
        if y<pbest_y[k]: pbest_y[k]=y; pbest[k]=P[k].copy()
        if y<gbest_y:    gbest_y=y;    gbest=P[k].copy()
    r1,r2=rng.rand(SWARM,D),rng.rand(SWARM,D)
    V=w_in*V+c1*r1*(pbest-P)+c2*r2*(gbest-P)
    V=np.clip(V,-0.25,0.25); P=np.clip(P+V,0,1)
    print('  pso iter %d/%d  gbest val %7.3f  swarm spread %.3f'
          %(it+1,ITERS,gbest_y,float(np.mean(np.std(P,axis=0)))))
print('PSO done in %.1f min | best val %.3f'%((time.time()-t0)/60,gbest_y))

## Stage 2 — re-decide the shortlist on fresh seeds

Stage 1 ranks on a single seed, so its argmin is biased low: the winner is partly the luckiest draw.
Stage 2 re-evaluates the top few from each search on seeds the search never saw, and the baseline
alongside them at identical budget. Only that comparison decides anything.

In [ ]:
T1=pd.DataFrame(TRIALS)
T1=T1[T1.tag.isin(['bo_init','bo','pso'])].copy()
short=T1.nsmallest(SHORTLIST*2,'val').drop_duplicates(subset=HPN).head(SHORTLIST)
print('shortlist (stage-1 validation):')
print(short[['tag','val','test_capped','test_raw','params']].to_string(index=False))

def kw_of(row):
    get=(lambda n: getattr(row,n)) if hasattr(row,'_fields') else (lambda n: row[n])
    out={}
    for s in SPACE:
        n,k=s[0],s[1]; v=get(n)
        out[n]=int(v) if k=='int' or n=='batch' or n=='kernel' or n=='use_pool' else float(v)
    return out

STAGE2=[]
cands=[('baseline',BASE)]+[('cand%d_%s'%(i+1,r.tag),kw_of(r)) for i,r in enumerate(short.itertuples())]
for name,kw in cands:
    vs,tc,tr_=[],[],[]
    for s in range(N_SEEDS2):
        evaluate(kw,SEED+100+s,'stage2:'+name)
        rr=TRIALS[-1]; vs.append(rr['val']); tc.append(rr['test_capped']); tr_.append(rr['test_raw'])
    STAGE2.append(dict(name=name,val=np.mean(vs),val_sd=np.std(vs,ddof=1),
                       capped=np.mean(tc),capped_sd=np.std(tc,ddof=1),
                       raw=np.mean(tr_),raw_sd=np.std(tr_,ddof=1),**kw))
    print('  %-18s val %6.3f+/-%.3f | capped %6.3f+/-%.3f | raw %6.3f+/-%.3f'
          %(name,STAGE2[-1]['val'],STAGE2[-1]['val_sd'],STAGE2[-1]['capped'],
            STAGE2[-1]['capped_sd'],STAGE2[-1]['raw'],STAGE2[-1]['raw_sd']))
S2=pd.DataFrame(STAGE2)

## Did search help? — the guard, and the diagnostic

In [ ]:
base=S2[S2['name']=='baseline'].iloc[0]
best=S2[S2['name']!='baseline'].nsmallest(1,'val').iloc[0]
d=best.capped-base.capped; se=math.sqrt(best.capped_sd**2+base.capped_sd**2)/math.sqrt(N_SEEDS2)
print('='*74)
print('%-22s %14s %14s'%('','capped-55','raw'))
print('%-22s %7.3f+/-%.3f %7.3f+/-%.3f'%('baseline',base.capped,base.capped_sd,base.raw,base.raw_sd))
print('%-22s %7.3f+/-%.3f %7.3f+/-%.3f'%('search winner',best.capped,best.capped_sd,best.raw,best.raw_sd))
print('%-22s %+7.3f  (paired SE %.3f)'%('difference',d,se))
print('='*74)
verdict = 'ADOPT' if (d < 0 and abs(d) > 2*se) else 'KEEP BASELINE (difference within noise)'
print('guard verdict:',verdict)
print()
A=T1.dropna(subset=['test_capped'])
rho,p=spearmanr(A.val,A.test_capped)
print('Spearman(validation, test-capped) over %d search trials: %+.3f  (p=%.4g)'%(len(A),rho,p))
print('  -> this is the number that says whether search CAN work here.')
print('  -> for reference, cross-cycle on the old representation gave +0.276.')
print()
print('did the searches converge back toward the baseline region?')
for tag,zs in [('BO',BO_Z[np.array(BO_Y)<999]),('PSO',np.array([gbest]))]:
    if len(zs):
        best_z=zs[int(np.argmin(np.array(BO_Y)[np.array(BO_Y)<999]))] if tag=='BO' else gbest
        print('   %-4s start (mean of far design) %.3f -> winner %.3f  from baseline'
              %(tag,np.mean([nrmsd(z,Z_BASE) for z in Zi]),nrmsd(best_z,Z_BASE)))
print()
print('per-hyperparameter Spearman against test-capped (where divergence comes from):')
for n in HPN:
    if A[n].nunique()>1:
        r,pp=spearmanr(A[n],A.test_capped)
        print('   %-12s rho %+6.3f  p %.3g %s'%(n,r,pp,'  *' if pp<0.05 else ''))

In [ ]:
floor=triv_cap
final = base if verdict.startswith('KEEP') else best
rows=[('MSTSDN published',3.95,None),
      ('ours, matched protocol + search',final.capped,final.raw),
      ('ours, matched protocol (v7)',5.137,10.268),
      ('ours, original protocol (v7 cell)',15.157,19.824),
      ('trivial floor',floor,triv_raw)]
print('%-38s %9s %9s %8s'%('','capped-55','raw','skill'))
for lab,c,r in rows:
    print('%-38s %9.3f %9s %+8.3f'%(lab,c,('%.3f'%r) if r is not None else '  --  ',1-c/floor))
print()
print('remaining gap to MSTSDN on their metric: %+.3f RMSE'%(final.capped-3.95))
S2.to_csv('ncmapss_v8_stage2.csv',index=False)
pd.DataFrame(TRIALS).to_csv('ncmapss_v8_all_trials.csv',index=False)
json.dump(dict(verdict=verdict,spearman=float(rho),baseline=base.to_dict(),
               winner=best.to_dict(),trivial_capped=floor,trivial_raw=triv_raw),
          open('ncmapss_v8_summary.json','w'),indent=2,default=str)
print('\nsaved ncmapss_v8_stage2.csv, ncmapss_v8_all_trials.csv, ncmapss_v8_summary.json')